# Fixed Effects Feasibility Analysis of Syndicated Loan Data

This notebook performs a detailed analysis of syndicated loan data (Dealscan) to assess which level of analysis has sufficient time variation and lender heterogeneity to conduct fixed-effects analysis. 

---

## Objectives

1. **Measure time variation and lender heterogeneity:**
   - For tranches (loan components).
   - For deals (overall facilities).
2. **Check fixed-effects feasibility for both deal and tranche level analysis.**
4. **Visualize participation over time for comparison.**

---

## Workflow Overview

✅ **1. Load and Parse Data**  
- Load the cleaned Dealscan dataset.
- Parse `Tranche_Active_Date` as a datetime field.

✅ **2. Create Firm-Year Variable**  
- Combine Borrower ID and Year into a "Firm-Year" identifier (e.g., `12345_2020`).
- This measures whether multiple banks participated in lending to the same borrower in the same year.

✅ **3. Measuring Key Metrics**
For each unit (tranche or deal), calculate:
- **% with multiple banks:** proportion of units involving >1 lender.
- **% with multiple active dates:** proportion of units amended or reactivated over time.
- **% meeting both criteria:** units with both lender heterogeneity and time variation.
- **% of firm-years with multiple banks:** borrower-years with more than one bank involved.

These metrics assess feasibility of fixed-effects estimation.

✅ **4. Tranche-Level Analysis**
- Treat each unique `LPC_Tranche_ID` as a unit.
- Compute metrics described above.
- **Filter tranches** that:
  - Have ≥2 unique active dates.
  - Involve >1 bank.
  - Include at least 1 U.S. bank.
- Summarize retained tranches and visualize per quarter.

✅ **5. Deal-Level Event Analysis(using both Tranche_Active_Date and Deal_Input_Date)**
- Define **Deal Events** as unique `(Deal_ID, Tranche_Active_Date)` pairs.
  - Each activation/amendment date of a deal is treated as a separate observation.
- Compute the same metrics as in tranche-level analysis.
- **Filter deals** that:
  - Have ≥2 unique event dates.
  - Involve >1 bank across all event dates.
  - Include at least 1 U.S. bank.
- Summarize retained deals and visualize per quarter.

---

## Outputs

- **Summary tables** showing counts of retained tranches or deals, unique borrowers, and unique U.S. banks.
- **Feasibility metrics** for fixed-effects regressions.
- **Plots per quarter** with annotated counts of U.S. banks and units.

### compare Tranche_Active_Date VS Deal_Input_Date

In [ ]:
# import pandas as pd

# # Load your dataset
# DATA_FILE = "2021jan_2024sept_cleaned.csv"

# # Columns of interest
# ID_COL = "LPC_Deal_ID"
# TRANCHE_DATE_COL = "Tranche_Active_Date"
# INPUT_DATE_COL = "Deal_Input_Date"

# # Load data
# df = pd.read_csv(DATA_FILE, parse_dates=[TRANCHE_DATE_COL, INPUT_DATE_COL])

# # Compute counts of unique dates per deal
# date_counts = (
#     df.groupby(ID_COL)
#     .agg(
#         n_tranche_active_dates=(TRANCHE_DATE_COL, "nunique"),
#         n_deal_input_dates=(INPUT_DATE_COL, "nunique")
#     )
#     .reset_index()
# )

# # Calculate:
# more_tranche_than_input = (date_counts["n_tranche_active_dates"] > date_counts["n_deal_input_dates"]).sum()
# equal_tranche_input = (date_counts["n_tranche_active_dates"] == date_counts["n_deal_input_dates"]).sum()
# less_tranche_than_input = (date_counts["n_tranche_active_dates"] < date_counts["n_deal_input_dates"]).sum()

# total_deals = date_counts.shape[0]

# # Percentages
# pct_more = (more_tranche_than_input / total_deals) * 100
# pct_equal = (equal_tranche_input / total_deals) * 100
# pct_less = (less_tranche_than_input / total_deals) * 100

# # Print results
# print(f"Total deals analyzed: {total_deals}\n")
# print("Counts:")
# print(f"- Deals where Tranche_Active_Date count > Deal_Input_Date count: {more_tranche_than_input}")
# print(f"- Deals where counts are equal: {equal_tranche_input}")
# print(f"- Deals where Tranche_Active_Date count < Deal_Input_Date count: {less_tranche_than_input}\n")

# print("Percentages:")
# print(f"- % Tranche > Input: {pct_more:.2f}%")
# print(f"- % Equal counts: {pct_equal:.2f}%")
# print(f"- % Tranche < Input: {pct_less:.2f}%")

# # Optional: save full results to CSV for inspection
# #date_counts.to_csv("deal_date_counts_comparison.csv", index=False)

Total deals analyzed: 28568

Counts:
- Deals where Tranche_Active_Date count > Deal_Input_Date count: 441
- Deals where counts are equal: 28127
- Deals where Tranche_Active_Date count < Deal_Input_Date count: 0

Percentages:
- % Tranche > Input: 1.54%
- % Equal counts: 98.46%
- % Tranche < Input: 0.00%

In [ ]:
# Tranche-Level Fixed Effects Feasibility Check (Clean Version)

import pandas as pd

# =============================
# CONFIGURATION
# =============================

DATA_FILE = "2021jan_2024sept_cleaned.csv"

# Column names
TRANCHE_ID = "LPC_Tranche_ID"
DEAL_ID = "LPC_Deal_ID"
DATE_COL = "Tranche_Active_Date"
BANK_ID = "Lender_Parent_Id"
BANK_COUNTRY = "Lender_Parent_Operating_Country"
BORROWER_ID = "Borrower_Id"

# =============================
# STEP 1: Load and Parse Data
# =============================

def load_data(filepath):
    """
    Load cleaned Dealscan data and parse Tranche_Active_Date as datetime.
    """
    df = pd.read_csv(filepath, parse_dates=[DATE_COL])
    return df

# =============================
# STEP 2: Create Firm-Year ID
# =============================

def create_firm_year(df):
    """
    Combine Borrower ID and Year into a Firm-Year string.
    """
    df["Year"] = df[DATE_COL].dt.year
    df["Firm_Year"] = df[BORROWER_ID].astype(str) + "_" + df["Year"].astype(str)
    return df

# =============================
# STEP 3: Fixed Effects Feasibility Check
# =============================

def summarize_fixed_effects(df, unit_col, date_col):
    """
    For each tranche, compute:
    1. % with multiple banks
    2. % with multiple active dates
    3. % with both
    4. % of firm-years with multiple banks
    """
    bank_counts = df.groupby(unit_col)[BANK_ID].nunique()
    date_counts = df.groupby(unit_col)[date_col].nunique()

    pct_multi_banks = (bank_counts > 1).mean() * 100
    pct_multi_dates = (date_counts > 1).mean() * 100
    pct_both = ((bank_counts > 1) & (date_counts > 1)).mean() * 100

    firm_year_counts = df.groupby("Firm_Year")[BANK_ID].nunique()
    pct_firm_year_multi_banks = (firm_year_counts > 1).mean() * 100

    print(f"\n🔍 Fixed Effects Feasibility Check (Tranche-Level):")
    print(f"1. Tranches with multiple banks       : {pct_multi_banks:.2f}%")
    print(f"2. Tranches with multiple dates       : {pct_multi_dates:.2f}%")
    print(f"3. Tranches with both banks & dates   : {pct_both:.2f}%")
    print(f"4. Firm-years with multiple banks     : {pct_firm_year_multi_banks:.2f}%")

    return bank_counts, date_counts

# =============================
# STEP 4: Tranche Filtering & Summary
# =============================

def filter_tranches(df):
    """
    Keep only tranches that have:
    - At least 2 active dates
    - Involve >1 bank
    - Include at least one U.S. bank
    """
    def is_valid(group):
        return (
            group[DATE_COL].nunique() >= 2 and
            group[BANK_ID].nunique() > 1 and
            "United States" in group[BANK_COUNTRY].values
        )

    df_filtered = df.groupby(TRANCHE_ID).filter(is_valid)
    return df_filtered

def report_summary(df):
    """
    Print summary stats of the filtered dataset.
    """
    n_tranches = df[TRANCHE_ID].nunique()
    n_borrowers = df[BORROWER_ID].nunique()
    n_us_banks = df[df[BANK_COUNTRY] == "United States"][BANK_ID].nunique()

    print("\n✅ Summary of Filtered Tranche-Level Dataset:")
    print(f"• Unique tranches  : {n_tranches}")
    print(f"• Unique borrowers : {n_borrowers}")
    print(f"• U.S. banks       : {n_us_banks}")

# =============================
# MAIN SCRIPT
# =============================

if __name__ == "__main__":
    # Load data
    df = load_data(DATA_FILE)

    # Create firm-year identifier
    df = create_firm_year(df)

    # Fixed effects feasibility (entire tranche dataset)
    bank_counts, date_counts = summarize_fixed_effects(df, TRANCHE_ID, DATE_COL)

    # Filter tranches based on feasibility criteria
    df_tranche_filtered = filter_tranches(df)

    # Summary of cleaned dataset
    report_summary(df_tranche_filtered)

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt

# # =============================
# # CONFIGURATION
# # =============================

# # File containing cleaned Dealscan data
# DATA_FILE = "2021jan_2024sept_cleaned.csv"

# # Column names in the dataset
# ID_COLUMNS = {
#     "tranche": "LPC_Tranche_ID",   # Unique identifier for each tranche
#     "deal": "LPC_Deal_ID"          # Unique identifier for each deal
# }
# DATE_COLUMN = "Tranche_Active_Date"          # Activation or amendment date of each tranche
# INPUT_DATE_COLUMN = "Deal_Input_Date"        # NEW: for second analysis
# BANK_COLUMN = "Lender_Parent_Id"             # ID of the parent bank
# US_BANK_COLUMN = "Lender_Parent_Operating_Country"  # Country of the parent bank
# BORROWER_COLUMN = "Borrower_Id"              # Borrower company ID

# # =============================
# # HELPER FUNCTIONS
# # =============================

# def load_data(filepath):
#     """
#     Load Dealscan data from CSV.
#     Automatically parses Tranche_Active_Date and Deal_Input_Date as datetime columns.
#     """
#     df = pd.read_csv(filepath, parse_dates=[DATE_COLUMN, INPUT_DATE_COLUMN])
#     return df

# def create_firm_year(df, date_col):
#     """
#     Create Firm-Year variable for grouping.
#     ---------------------------------------------------
#     This combines:
#         - Borrower_Id (the firm)
#         - Year extracted from the date column
#     Into a single string like "12345_2020".
#     This allows us to count:
#         - How many different banks lent to the same borrower
#           in the same year.
#     """
#     df["Year"] = df[date_col].dt.year
#     df["Firm_Year"] = df[BORROWER_COLUMN].astype(str) + "_" + df["Year"].astype(str)
#     return df

# def summarize_fixed_effects(df, unit_col, date_col):
#     """
#     Compute feasibility metrics to assess whether fixed effects regressions
#     are possible with this dataset.
#     ---------------------------------------------------
#     For each unit (tranche or deal), calculates:
#         1. % with >1 unique bank involved
#         2. % with >1 unique active date
#         3. % meeting both conditions (multiple banks & dates)
#     Separately, computes:
#         4. % of Firm-Year pairs with multiple banks
#     These statistics help evaluate:
#         - Whether there is sufficient variation within units and over time
#           to identify causal effects of deposits on lending.
#     """
#     # Count # of unique banks per unit
#     bank_counts = df.groupby(unit_col)[BANK_COLUMN].nunique()
#     pct_multiple_banks = (bank_counts > 1).mean() * 100

#     # Count # of unique dates per unit
#     date_counts = df.groupby(unit_col)[date_col].nunique()
#     pct_multiple_dates = (date_counts > 1).mean() * 100

#     # Units with both multiple banks and multiple dates
#     combined = (bank_counts > 1) & (date_counts > 1)
#     pct_combined = combined.mean() * 100

#     # Firm-Year pairs with multiple banks
#     firm_year_counts = df.groupby("Firm_Year")[BANK_COLUMN].nunique()
#     pct_firm_year_multiple_banks = (firm_year_counts > 1).mean() * 100

#     # Print summary
#     print(f"\n🔍 Fixed Effects Feasibility Check ({unit_col}) using {date_col}:")
#     print(f"1. Units with multiple banks               : {pct_multiple_banks:.2f}%")
#     print(f"2. Units with multiple active dates        : {pct_multiple_dates:.2f}%")
#     print(f"3. Units with both multiple banks & dates  : {pct_combined:.2f}%")
#     print(f"4. Firm-year pairs with multiple banks     : {pct_firm_year_multiple_banks:.2f}%")

#     return bank_counts, date_counts

# def plot_counts_per_quarter(df, unit_col, date_col, title_prefix):
#     """
#     Generate a dual-axis plot showing:
#     ---------------------------------------------------
#     - Number of unique U.S. banks per quarter (bar chart)
#     - Number of unique units (tranches or deals) per quarter (line chart)
#     Each bar and point is labeled with the count for clarity.
#     """
#     # Create Quarter variable for grouping
#     df["Quarter"] = df[date_col].dt.to_period("Q").astype(str)

#     # Count U.S. banks per quarter
#     us_banks_q = (
#         df[df[US_BANK_COLUMN] == "United States"]
#         .groupby("Quarter")[BANK_COLUMN]
#         .nunique()
#     )

#     # Count units (tranches or deal events) per quarter
#     units_q = df.groupby("Quarter")[unit_col].nunique()

#     # Create the figure
#     fig, ax1 = plt.subplots(figsize=(12, 6))

#     # Bar plot for U.S. banks
#     bars = ax1.bar(us_banks_q.index, us_banks_q.values, alpha=0.6, label="U.S. Banks")
#     ax1.set_ylabel("Number of U.S. Banks")
#     ax1.set_title(f"{title_prefix}: U.S. Banks and {unit_col} per Quarter")
#     ax1.tick_params(axis="x", rotation=45)

#     # Annotate each bar with count
#     for bar in bars:
#         height = bar.get_height()
#         ax1.annotate(
#             f"{int(height)}",
#             xy=(bar.get_x() + bar.get_width() / 2, height),
#             xytext=(0, 3),
#             textcoords="offset points",
#             ha="center", va="bottom", fontsize=8
#         )

#     # Line plot for unit counts
#     ax2 = ax1.twinx()
#     line = ax2.plot(units_q.index, units_q.values, color="red", marker="o", label=f"{unit_col} count")
#     ax2.set_ylabel(f"Number of {unit_col}")

#     # Annotate each point with count
#     for x, y in zip(units_q.index, units_q.values):
#         ax2.annotate(
#             f"{int(y)}",
#             xy=(x, y),
#             xytext=(0, 5),
#             textcoords="offset points",
#             ha="center", va="bottom",
#             fontsize=8, color="red"
#         )

#     # Add legend and layout adjustments
#     fig.legend(loc="upper left", bbox_to_anchor=(0.1, 0.9))
#     plt.tight_layout()
#     plt.show()

# def report_summary(df, unit_col):
#     """
#     Print a concise summary of the dataset after filtering.
#     ---------------------------------------------------
#     Includes:
#         - Number of unique units (tranches or deals)
#         - Number of unique borrowers
#         - Number of unique U.S. banks
#     """
#     n_units = df[unit_col].nunique()
#     n_borrowers = df[BORROWER_COLUMN].nunique()
#     n_us_banks = df[df[US_BANK_COLUMN] == "United States"][BANK_COLUMN].nunique()

#     print("\n✅ Summary:")
#     print(f"• Unique {unit_col}: {n_units}")
#     print(f"• Unique borrowers: {n_borrowers}")
#     print(f"• U.S. banks: {n_us_banks}")

# =============================
# MAIN WORKFLOW
# =============================


In [ ]:
# if __name__ == "__main__":
#     # Load the full dataset
#     df_all = load_data(DATA_FILE)

#     # ============================================
#     # ========== TRANCHE-LEVEL ANALYSIS ==========
#     # ============================================
#     print("\n================ Tranche-Level Analysis ================")

#     df_tranche = create_firm_year(df_all.copy(), DATE_COLUMN)

#     # Compute fixed effects feasibility
#     bank_counts_t, date_counts_t = summarize_fixed_effects(
#         df_tranche,
#         ID_COLUMNS["tranche"],
#         DATE_COLUMN
#     )

#     # Prepare deal-level for hierarchy check
#     df_deal_events = create_firm_year(df_all.copy(), DATE_COLUMN)
#     bank_counts_d, date_counts_d = summarize_fixed_effects(
#         df_deal_events,
#         ID_COLUMNS["deal"],
#         DATE_COLUMN
#     )

#     # Hierarchy check: map each tranche to its parent deal
#     print("\n🔍 HIERARCHY CHECK: TRANCHE vs DEAL")
#     tranche_to_deal = df_tranche[[ID_COLUMNS["tranche"], ID_COLUMNS["deal"]]].drop_duplicates().set_index(ID_COLUMNS["tranche"])

#     comparison = (
#         pd.DataFrame({
#             "Banks_per_Tranche": bank_counts_t,
#             "TimeObs_per_Tranche": date_counts_t
#         })
#         .merge(tranche_to_deal, left_index=True, right_index=True)
#         .merge(
#             pd.DataFrame({
#                 "Banks_per_Deal": bank_counts_d,
#                 "TimeObs_per_Deal": date_counts_d
#             }),
#             left_on=ID_COLUMNS["deal"],
#             right_index=True,
#             how='left'
#         )
#     )

#     bank_violations = comparison["Banks_per_Tranche"] > comparison["Banks_per_Deal"]
#     time_violations = comparison["TimeObs_per_Tranche"] > comparison["TimeObs_per_Deal"]
#     print(f"🚨 Tranches with MORE banks than parent deal: {bank_violations.sum()} ({bank_violations.mean()*100:.2f}%)")
#     print(f"🚨 Tranches with MORE time obs than parent deal: {time_violations.sum()} ({time_violations.mean()*100:.2f}%)")

#     # Tranche-level summary
#     df_tranche_filtered = df_tranche.groupby(ID_COLUMNS["tranche"]).filter(
#         lambda x: x[DATE_COLUMN].nunique() >= 2 and
#                   x[BANK_COLUMN].nunique() > 1 and
#                   "United States" in x[US_BANK_COLUMN].values
#     )
#     report_summary(df_tranche_filtered, ID_COLUMNS["tranche"])

#     # ============================================
#     # == DEAL-LEVEL ANALYSIS (Tranche_Active_Date)
#     # ============================================
#     print("\n================ Deal-Level Analysis (Tranche_Active_Date) ================")

#     df_deal_active = create_firm_year(df_all.copy(), DATE_COLUMN)

#     # Fixed effects feasibility
#     summarize_fixed_effects(df_deal_active, ID_COLUMNS["deal"], DATE_COLUMN)

#     # Deal-level summary
#     df_deal_active_filtered = df_deal_active.groupby(ID_COLUMNS["deal"]).filter(
#         lambda x: x[DATE_COLUMN].nunique() >= 2 and
#                   x[BANK_COLUMN].nunique() > 1 and
#                   "United States" in x[US_BANK_COLUMN].values
#     )
#     report_summary(df_deal_active_filtered, ID_COLUMNS["deal"])

#     # ============================================
#     # == DEAL-LEVEL ANALYSIS (Deal_Input_Date)
#     # ============================================
#     print("\n================ Deal-Level Analysis (Deal_Input_Date) ================")

#     df_deal_input = create_firm_year(df_all.copy(), INPUT_DATE_COLUMN)

#     # Fixed effects feasibility
#     summarize_fixed_effects(df_deal_input, ID_COLUMNS["deal"], INPUT_DATE_COLUMN)

#     # Deal-level summary
#     df_deal_input_filtered = df_deal_input.groupby(ID_COLUMNS["deal"]).filter(
#         lambda x: x[INPUT_DATE_COLUMN].nunique() >= 2 and
#                   x[BANK_COLUMN].nunique() > 1 and
#                   "United States" in x[US_BANK_COLUMN].values
#     )
#     report_summary(df_deal_input_filtered, ID_COLUMNS["deal"])

#     # ============================================
#     # ================ PLOTTING ==================
#     # ============================================
#     plot_counts_per_quarter(df_tranche_filtered, ID_COLUMNS["tranche"], DATE_COLUMN, "Tranche-Level")
#     plot_counts_per_quarter(df_deal_active_filtered, ID_COLUMNS["deal"], DATE_COLUMN, "Deal-Level Events (Tranche_Active_Date)")
#     plot_counts_per_quarter(df_deal_input_filtered, ID_COLUMNS["deal"], INPUT_DATE_COLUMN, "Deal-Level Events (Deal_Input_Date)")

In [ ]:
deal_tranche_counts = df_all.groupby("LPC_Deal_ID")["LPC_Tranche_ID"].nunique()
print(f"Deals with only 1 tranche: {(deal_tranche_counts == 1).mean()*100:.2f}%")

	•	Every LPC_Deal_ID maps to exactly one Deal_PermID
	•	Every Deal_PermID maps to exactly one LPC_Deal_ID


In [ ]:
#check LPC_Deal_ID VS Deal_PermID
# DATA_FILE = "2021jan_2024sept_cleaned.csv"

# # Load a portion of the large filer
# df = pd.read_csv(DATA_FILE, usecols=["LPC_Deal_ID", "Deal_PermID"])

# # Drop missing values just in case
# df = df.dropna(subset=["LPC_Deal_ID", "Deal_PermID"])

# # Convert to string if needed
# df["LPC_Deal_ID"] = df["LPC_Deal_ID"].astype(str)
# df["Deal_PermID"] = df["Deal_PermID"].astype(str)

# # 1. How many LPC_Deal_IDs per Deal_PermID?
# dealperm_to_lpc_counts = df.groupby("Deal_PermID")["LPC_Deal_ID"].nunique()

# # 2. How many Deal_PermIDs per LPC_Deal_ID?
# lpc_to_dealperm_counts = df.groupby("LPC_Deal_ID")["Deal_PermID"].nunique()

# # 3. Print summaries
# print("\n🔍 Mapping from Deal_PermID → LPC_Deal_ID:")
# print(f"• Unique Deal_PermIDs: {dealperm_to_lpc_counts.shape[0]}")
# print(f"• Deal_PermIDs mapping to >1 LPC_Deal_ID: {(dealperm_to_lpc_counts > 1).sum()}")

# print("\n🔍 Mapping from LPC_Deal_ID → Deal_PermID:")
# print(f"• Unique LPC_Deal_IDs: {lpc_to_dealperm_counts.shape[0]}")
# print(f"• LPC_Deal_IDs mapping to >1 Deal_PermID: {(lpc_to_dealperm_counts > 1).sum()}")

# # Optional: show a few examples of non-1-to-1 mappings
# print("\nExamples where Deal_PermID maps to multiple LPC_Deal_IDs:")
# print(df[df["Deal_PermID"].isin(dealperm_to_lpc_counts[dealperm_to_lpc_counts > 1].index)].head(10))

# print("\nExamples where LPC_Deal_ID maps to multiple Deal_PermIDs:")
# print(df[df["LPC_Deal_ID"].isin(lpc_to_dealperm_counts[lpc_to_dealperm_counts > 1].index)].head(10))

# Old Code

In [ ]:
# import pandas as pd
# df = pd.read_csv("2021jan_2024sept_cleaned.csv")
# # Tranche-level relevant fields (cleaned names)
# tranche_fields = [
#     'LPC_Tranche_ID',
#     'Tranche_Amount',
#     'Tranche_Active_Date',
#     'Tranche_Maturity_Date',
#     'Tranche_Amended',
#     'Tranche_O_A',
#     'Tranche_Currency',
    
#     'LPC_Deal_ID',
#     'Deal_Amount',
#     'Deal_Active_Date',
#     'Deal_Input_Date',
#     'Deal_Amended',
#     'Phase',
#     'Deal_Purpose',

#     'Borrower_Id',
#     'Borrower_Name',
#     'Major_Industry_Group',

#     'Lender_Parent_Id',
#     'Lender_Name',
#     'Lender_Id',
#     'Lender_Parent_Name',
#     'Lender_Parent_Operating_Country'
# ]

# # Filter the DataFrame
# df_tranche = df[tranche_fields]

# structure panel quarter data

### Add quarter

In [ ]:
# # Ensure date columns are in datetime format
# df_tranche["Tranche_Active_Date"] = pd.to_datetime(df_tranche["Tranche_Active_Date"], errors="coerce")
# df_tranche["Tranche_Maturity_Date"] = pd.to_datetime(df_tranche["Tranche_Maturity_Date"], errors="coerce")

# # Rename for consistency with previous code
# df_tranche = df_tranche.copy()

# # 📆 Step 1: Define quarterly periods (from Jan 2021 to Sep 2024)
# quarterly_periods = pd.date_range(start="2021-01-01", end="2024-09-30", freq="Q")

In [ ]:
# # 🧺 Step 2: Create a list to collect quarterly panel records
# panel_records = []

# # 🔁 Step 3: Loop through each quarterly period
# for quarter_end in quarterly_periods:
#     # Calculate quarter start
#     quarter_start = pd.Timestamp(quarter_end) - pd.DateOffset(months=2) - pd.DateOffset(days=quarter_end.day - 1)
    
#     # 🧹 Step 4: Filter rows where tranche is active during this quarter
#     active_tranches = df_tranche[
#         (df_tranche["Tranche_Active_Date"] <= quarter_end) &
#         (df_tranche["Tranche_Maturity_Date"] >= quarter_start)
#     ].copy()

#     # 🏷️ Step 5: Add quarter labels
#     active_tranches["quarter_start"] = quarter_start
#     active_tranches["quarter_end"] = quarter_end

#     # Collect into list
#     panel_records.append(active_tranches)

# # 📊 Step 6: Combine all quarters into a single panel dataframe
# df_panel = pd.concat(panel_records).reset_index(drop=True)

# # ✅ Step 7: Preview results
# print(f"Total records in quarterly panel: {len(df_panel)}")
# print("Sample records:")
# display(df_panel.head())


### aggregate

In [ ]:
# # Ensure datetime columns are properly formatted
# df_panel["Tranche_Active_Date"] = pd.to_datetime(df_panel["Tranche_Active_Date"])
# df_panel["Tranche_Maturity_Date"] = pd.to_datetime(df_panel["Tranche_Maturity_Date"])
# df_panel["quarter_start"] = pd.to_datetime(df_panel["quarter_start"])
# df_panel["quarter_end"] = pd.to_datetime(df_panel["quarter_end"])

# # Step 1: Add flags for new and matured tranches
# df_panel["is_new_tranche"] = (df_panel["Tranche_Active_Date"] >= df_panel["quarter_start"]) & (df_panel["Tranche_Active_Date"] <= df_panel["quarter_end"])
# df_panel["is_matured_tranche"] = (df_panel["Tranche_Maturity_Date"] >= df_panel["quarter_start"]) & (df_panel["Tranche_Maturity_Date"] <= df_panel["quarter_end"])

# # Step 2: Group and aggregate by quarter
# quarterly_bank_summary = df_panel.groupby([
#     "quarter_end",
#     "Lender_Parent_Operating_Country",
#     "Lender_Parent_Id",
#     "Lender_Parent_Name",
#     'Lender_Id',
#     "Lender_Name"
# ]).agg(
#     total_active_tranches=("LPC_Tranche_ID", "nunique"),
#     total_new_tranches=("is_new_tranche", "sum"),
#     total_matured_tranches=("is_matured_tranche", "sum"),
#     total_amount_outstanding=("Tranche_Amount", "sum")
# ).reset_index()

# # Preview result
# quarterly_bank_summary

In [ ]:
# quarterly_bank_summary.to_csv("tranche_explore/Full_Data_quarterly_bank_summary.csv", index=False)

# Fixed-Effects Feasibility Checks for both Deals and Tranches

✅ **Results Using `Lender_Id`** — *Summary of Qualified Tranches*:

1. **Retained tranches**: 4,279  
2. **Unique borrower companies**: 2,201  
3. **Unique U.S. banks involved**: 521

In [ ]:
# # Step 1: Group by LPC_Tranche_ID and count the number of unique lenders per tranche
# lender_counts = df_tranche.groupby("LPC_Tranche_ID")["Lender_Parent_Id"].nunique()

# # Step 2: Count how many tranches have more than one unique lender
# multi_lender_tranches = (lender_counts > 1).sum()
# multi_lender_tranches

# # Step 3: Total number of unique tranches
# total_tranches = lender_counts.shape[0]

# # Step 4: Calculate the percentage
# percentage_multi_lender = (multi_lender_tranches / total_tranches) * 100

# # Step 5: Print summary
# print(f"Total unique tranches: {total_tranches}")
# print(f"Tranches with more than one lender by Lender_Parent_Id: {multi_lender_tranches}")
# print(f"Percentage of multi-lender tranches by Lender_Parent_Id: {percentage_multi_lender:.2f}%")

# # Create firm-year key
# df_tranche['Year'] = df_tranche['Tranche_Active_Date'].dt.year
# df_tranche['Firm_Year'] = df_tranche['Borrower_Id'].astype(str) + "_" + df_tranche['Year'].astype(str)
# df_tranche['Firm_Year']

# # === Check 1: % of tranches with more than one bank ===
# tranche_bank_counts = df_tranche.groupby('LPC_Tranche_ID')['Lender_Id'].nunique()
# pct_tranches_multiple_banks = (tranche_bank_counts > 1).mean() * 100

# # === Check 2: % of tranches with more than one activation date (i.e., amendments) ===
# tranche_date_counts = df_tranche.groupby('LPC_Tranche_ID')['Tranche_Active_Date'].nunique()
# pct_tranches_multiple_dates = (tranche_date_counts > 1).mean() * 100

# # === Check 3: % of tranches that meet both conditions ===
# tranche_combined = (tranche_bank_counts > 1) & (tranche_date_counts > 1)
# pct_combined = tranche_combined.mean() * 100

# # === Check 4: % of firm-year pairs with more than one lender ===
# firm_year_bank_counts = df_tranche.groupby('Firm_Year')['Lender_Id'].nunique()
# pct_firm_year_multiple_banks = (firm_year_bank_counts > 1).mean() * 100

# # === Summary of results ===
# print("🔍 Fixed Effects Feasibility Check (based on Lender_Id):")
# print(f"1. Tranches with multiple banks                    : {pct_tranches_multiple_banks:.2f}%")
# print(f"2. Tranches with multiple active dates             : {pct_tranches_multiple_dates:.2f}%")
# print(f"3. Tranches with both multiple banks and dates     : {pct_combined:.2f}%")
# print(f"4. Firm-year pairs with multiple participating banks: {pct_firm_year_multiple_banks:.2f}%")

# # Step 2: Tranche-level summary for filtering
# tranche_summary = df_tranche.groupby("LPC_Tranche_ID").agg(
#     obs_count=("Tranche_Active_Date", "nunique"),  # active date variation
#     unique_banks=("Lender_Parent_Id", "nunique"),  # distinct banks
#     has_us_bank=("Lender_Parent_Operating_Country", lambda x: "United States" in x.values)  # at least one US bank
# )


# # Step 3: Apply filtering conditions
# valid_tranches = tranche_summary[
#     (tranche_summary["obs_count"] >= 2) &
#     (tranche_summary["unique_banks"] > 1) &
#     (tranche_summary["has_us_bank"])
# ].index

# # Step 4: Subset the original dataset
# quanlify_tranches = df_tranche[df_tranche["LPC_Tranche_ID"].isin(valid_tranches)]

# # Step 5: Final counts
# num_retained_tranches = quanlify_tranches["LPC_Tranche_ID"].nunique()
# num_borrowers = quanlify_tranches["Borrower_Id"].nunique()
# num_us_banks = quanlify_tranches[quanlify_tranches["Lender_Parent_Operating_Country"] == "United States"]["Lender_Parent_Id"].nunique()

# # Step 6: Print results
# print("✅ Summary of Qualified Tranches:")
# print(f"1. Retained tranches: {num_retained_tranches}")
# print(f"2. Unique borrower companies: {num_borrowers}")
# print(f"3. Unique U.S. banks involved: {num_us_banks}")

# import matplotlib.pyplot as plt
# # === 1. Basic counts ===
# num_tranches = quanlify_tranches['LPC_Tranche_ID'].nunique()
# num_borrowers = quanlify_tranches['Borrower_Id'].nunique()
# num_us_banks = quanlify_tranches[quanlify_tranches['Lender_Parent_Operating_Country'] == "United States"]["Lender_Parent_Id"].nunique()
# num_total_banks = quanlify_tranches['Lender_Parent_Id'].nunique()

# print("📊 Qualified Tranche Summary:")
# print(f"• Unique tranches: {num_tranches}")
# print(f"• Unique borrowers: {num_borrowers}")
# print(f"• U.S. parent banks: {num_us_banks}")
# print(f"• Total distinct parent banks: {num_total_banks}")

# # === 2. Top borrowers ===
# print("\n🏦 Top 10 borrowers by number of tranches:")
# print(quanlify_tranches['Borrower_Name'].value_counts().head(10))
# print("\n🏦 Top 10 Lenders by number of tranches:")
# print(quanlify_tranches['Lender_Name'].value_counts().head(10))

# # === 3. Top U.S. banks involved ===
# top_us_banks = quanlify_tranches[quanlify_tranches['Lender_Parent_Operating_Country'] == "United States"]['Lender_Parent_Name'].value_counts().head(10)
# print("\n🇺🇸 Top 10 U.S. banks by participation count:")
# print(top_us_banks)

# # === 4. Distribution: # of banks per tranche ===
# bank_counts = quanlify_tranches.groupby('LPC_Tranche_ID')['Lender_Parent_Id'].nunique()

# plt.figure(figsize=(8, 5))
# plt.hist(bank_counts, bins=range(2, bank_counts.max() + 2), edgecolor='black')
# plt.title("Distribution of Number of Banks per Tranche")
# plt.xlabel("Number of Banks")
# plt.ylabel("Number of Tranches")
# plt.grid(True)
# plt.tight_layout()
# plt.show()

# # === 5. Time coverage (optional) ===
# quanlify_tranches['Year'] = pd.to_datetime(quanlify_tranches['Tranche_Active_Date']).dt.year
# print("\n📅 Tranche count by year:")
# print(quanlify_tranches['Year'].value_counts().sort_index())